# Algoritmos de optimización - Seminario<br>
Nombre y Apellidos:   

*   Daniela Alejandra Escobar Suárez
*   Iker Zubizarreta Iturbe


Url: [https://github.com/DaniEscUCM/03MIAR-Algoritmos-de-optimizaci-n.git](https://github.com/DaniEscUCM/03MIAR-Algoritmos-de-optimizaci-n.git)<br>
Problema:
> 1. Sesiones de doblaje <br>
>2. Organizar los horarios de partidos de La Liga<br>
>3. Combinar cifras y operaciones

Descripción del problema:<br><br>
El problema consiste en analizar el siguiente problema y diseñar un algoritmo que lo resuelva.<br>
Disponemos de las 9 cifras del 1 al 9 (excluimos el cero) y de los 4 signos básicos de las operaciones fundamentales: suma(+), resta(-), multiplicación(\*) y división(/). Debemos combinarlos alternativamente sin repetir ninguno de ellos para obtener una cantidad dada. Un ejemplo sería para obtener el 4: 4+2-6/3\*1 = 4
<br>
Debe analizarse el problema para encontrar todos los valores enteros posibles planteando las siguientes cuestiones:<br>


*   ¿Qué valor máximo y mínimo se pueden obtener según las condiciones del problema?
*   ¿Es posible encontrar todos los valores enteros posibles entre dicho mínimo y máximo ?

**Nota: Es posible usar la función de python “eval” para evaluar una expresión**

<br>
(*) La respuesta es obligatoria





                                        

## (*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?<br>



## ¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones?




### **Respuesta:**


**Número de posibilidades sin tener en cuenta las restricciones:**</br>
Si quitamos la restricción de no poder repetir los símbolos y los números (dejo la restricción de alternarlos y máximo de 4 símbolos, para que tenga sentido).</br>
Por parte de los números, tenemos 9 posibles valores, que asignar en 5 posiciones que se pueden repetir: $n^k=9^5=59,049$

Por parte de los símbolos, tenemos 4 símbolos que podemos asignar a 4 posiciones que se pueden repetir: $n^k=4^4=256$

Total de posibilidades: $59049*256=15,116,544$

Hay exactamente 15 millones de posibilidades</br></br>


**Número de posibilidades teniendo en cuenta las restricciones:**

Agregamos la restricción de que no se repitan los números ni los símbolos.

Por parte de los números, tenemos 9 posibles valores, que asignar en 5 posiciones que **no** se pueden repetir: $ \frac{n!}{(n-k)!}= \frac{9!}{4!}=15,120$

Por parte de los símbolos, tenemos 4 símbolos que podemos asignar a 4 posiciones que **no** se pueden repetir: $ n!=4!=24$

otal de posibilidades: $15,120*24=362,880$

Hay exactamente 362,880 posibilidades


## Modelo para el espacio de soluciones<br>
## (*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, arguméntalo)


### Respuesta

**Estructura de datos elegida: lista (orden) + sets (control de repetidos)**

Representamos cada expresión candidata como una **lista de cadenas** (`secuencia`), donde cada elemento es, alternativamente, una cifra o un operador, en el mismo orden en que van a aparecer en la expresión final. Esto nos permite construir la expresión con `''.join(secuencia)` y pasarla directamente a `eval()`.

Junto a la lista mantenemos dos **sets** auxiliares, `numeros_usados` y `operadores_usados`, que sirven únicamente para comprobar en O(1) si una cifra o un operador ya se ha utilizado, sin tener que recorrer la secuencia entera cada vez.

*¿Por qué esta combinación y no otra?*

- Al principio, como en la fuerza bruta del notebook, se puede pensar en generar con `itertools.permutations` todas las permutaciones de cifras y, por separado, todas las de operadores. El problema es que esto obliga a tener la permutación **completa** antes de poder evaluar nada: no hay forma de cortar una rama a mitad de camino, así que no sirve para hacer poda.
- Necesitamos una **lista**, no una tupla, porque el backtracking añade y quita el último elemento constantemente (`append` / `pop`) según avanza y retrocede por el árbol de búsqueda — son operaciones O(1) al final de una lista.
- Usamos **sets** en vez de listas para "usados" porque esa comprobación (¿esta cifra/operador ya está en la expresión?) se ejecuta decenas de miles de veces durante la búsqueda; en una lista sería O(n) cada vez, en un set es O(1).
- No hace falta ninguna estructura más compleja (árbol explícito, cola de prioridad...) porque el propio orden de las llamadas recursivas del backtracking ya actúa como un recorrido en profundidad (DFS) sobre el árbol de posibilidades: no necesitamos materializarlo en memoria, solo recorrerlo.

Un ejemplo mínimo de cómo se usa esta estructura mientras se construye una expresión:

In [2]:
# Pequeño ejemplo de la estructura de datos usada en el backtracking:
# una lista (orden) + dos sets (control de repetidos en O(1))
secuencia = []
numeros_usados = set()
operadores_usados = set()

for token in ['4', '+', '2', '-', '6', '/', '3']:
    es_numero = token not in ['+', '-', '*', '/']
    usados = numeros_usados if es_numero else operadores_usados
    valor = int(token) if es_numero else token
    print(f"¿'{token}' ya usado? {valor in usados}  ->  se añade")
    secuencia.append(token)
    usados.add(valor)

print("\nSecuencia construida:", secuencia)
print("Expresión evaluable con eval():", ''.join(secuencia))
print("Números usados:", numeros_usados, " | Operadores usados:", operadores_usados)

¿'4' ya usado? False  ->  se añade
¿'+' ya usado? False  ->  se añade
¿'2' ya usado? False  ->  se añade
¿'-' ya usado? False  ->  se añade
¿'6' ya usado? False  ->  se añade
¿'/' ya usado? False  ->  se añade
¿'3' ya usado? False  ->  se añade

Secuencia construida: ['4', '+', '2', '-', '6', '/', '3']
Expresión evaluable con eval(): 4+2-6/3
Números usados: {2, 3, 4, 6}  | Operadores usados: {'+', '/', '-'}


## Según el modelo para el espacio de soluciones<br>
## (*)¿Cual es la función objetivo?

## (*)¿Es un problema de maximización o minimización?

## Respuesta


La función objetivo es el valor de la expresión construida, obtenido mediante `eval()`, que se compara con el valor objetivo para comprobar si ambos coinciden. En este caso no se trata de un problema de maximización ni de minimización, sino de un problema de decisión o satisfacción, ya que el objetivo es determinar si existe una expresión válida cuyo resultado sea exactamente el valor buscado. La ramificación y poda emplea cotas para descartar aquellas ramas desde las que el objetivo ya no puede alcanzarse.

## Diseña un algoritmo para resolver el problema por fuerza bruta

### Respuesta

In [10]:

def evaluar_valor(lista):
  """
  Evalua la expresión juntando los elementos de la lista y la función eval,
  evaluando la expresión dada. Se asume que la expresión es correcta. Por ejemplo,
  ['2','+','2'] -> 4
  """
  expresion=''.join(lista)
  return eval(expresion)

# Pruebas de ejemplo
#4+2-6/3*1 = 4
ejem_1 = ['4','+','2','-','6','/','3','*','1']
print(evaluar_valor(ejem_1))
# el valor -9*8 va a hacer cualquier valor pequeño
ejem_2 = ['2','/','7','+','1','-','9','*','8']
print(evaluar_valor(ejem_2))
ejem_2 = ['2','+','1','-','9','*','8','/','1']
print(evaluar_valor(ejem_2))
# el valor +9*8 va a hacer cualquier valor grande
ejem_3 = ['7','/','1','+','9','*','8','-','2']
print(evaluar_valor(ejem_3))

def permutar(lista):
    """
    Calcula todas las posibles permutaciones de la lista dada y lo retorna en
    una lista.
    """
    resultado = [[]]

    for x in lista:
        nuevo = []
        for p in resultado:
            for i in range(len(p) + 1):
                nuevo.append(p[:i] + [x] + p[i:])
        resultado = nuevo

    return resultado

def combinaciones():
  """
  Calcula las posibles combinaciones de los números de 1 al 9, el orden no
  importa y no se admiten repeticiones.
  """
  combinaciones = []

  for a in range(1, 6):
      for b in range(a + 1, 7):
          for c in range(b + 1, 8):
              for d in range(c + 1, 9):
                  for e in range(d + 1, 10):
                      combinaciones.append([str(a), str(b), str(c), str(d), str(e)])
  return combinaciones

def minimo_maximo_simbolo(numeros):
  """
  Calcula los valores máximos y mínimos con su correspondiente expresión
  dados unos números, que se les asigna los símbolos.
  """
  simbolos=permutar(['+','-','*','/'])
  min_exp=[]
  min=float('inf')
  max_exp=[]
  max=-float('inf')
  for sim in simbolos:
    expresion=[x for par in zip(numeros, sim) for x in par]
    expresion.append(numeros[-1])
    resul=evaluar_valor(expresion)

    if min > resul:
      min=resul
      min_exp=expresion

    if max < resul:
      max=resul
      max_exp=expresion


  return (min,min_exp,max,max_exp)

def max_min():
  """
  Busca el valor mínimo y máximo de todas las posibles combinaciones del
  problema.
  """
  combinaciones_num = combinaciones()
  min_exp=[]
  min=float('inf')
  max_exp=[]
  max=-float('inf')
  for combi in combinaciones_num:
    numeros=permutar(combi)
    for combinacion in numeros:
      resul_min,expresion_min,resul_max,expresion_max=minimo_maximo_simbolo(combinacion[:5])

      if min > resul_min:
        min=resul_min
        min_exp=expresion_min

      if max < resul_max:
        max=resul_max
        max_exp=expresion_max

  min_resul=''.join(min_exp)
  print(f"Minimo con valor {min} es {min_resul}")
  max_resul=''.join(max_exp)
  print(f"Maximo con valor {max} es {max_resul}")

max_min()

4.0
-70.71428571428571
-69.0
77.0
Minimo con valor -70.71428571428571 es 2/7-9*8+1
Maximo con valor 78.83333333333333 es 9*8+7-1/6


In [14]:
import math

def existe_valor_fuerza_bruta(objetivo):
    """Prueba, por fuerza bruta, si `objetivo` se puede obtener exactamente."""
    combinaciones_num = combinaciones()
    permutaciones_simbolos = permutar(['+', '-', '*', '/'])
    for combi in combinaciones_num:
        for numeros in permutar(combi):
            numeros = numeros[:5]
            for sim in permutaciones_simbolos:
                expresion = [x for par in zip(numeros, sim) for x in par]
                expresion.append(numeros[-1])
                if evaluar_valor(expresion) == objetivo:
                    return True, ''.join(expresion)
    return False, None

# Nota: recorrer TODOS los enteros con esta versión de fuerza bruta es muy costoso
# (362.880 evaluaciones por cada entero). Lo resolveremos de forma eficiente
# con el algoritmo de ramificación y poda de la siguiente sección; aquí dejamos
# solo el mecanismo de comprobación para un valor concreto, a modo de ejemplo.
print(existe_valor_fuerza_bruta(4))
print(existe_valor_fuerza_bruta(100))  # fuera de rango -> no deberia encontrarlo


# Todos los valores entre el mínimo y el máximo tienen solución en el problema?
for i in range(-70,79):
  existe, valor = existe_valor_fuerza_bruta(i)
  if(not existe):
    print(f'No existe solución para: {i}')

(True, '5-3+4/2*1')
(False, None)
No existe solución para: -70
No existe solución para: 78


Con la ayuda de la solución con fuerza bruta podemos responder a las incógnitas ¿Qué valor máximo y mínimo se pueden obtener según las condiciones del problema? y ¿Es posible encontrar todos los valores enteros posibles entre dicho mínimo y máximo?

El valor entero mínimo que se puede calcular en este problema es -69 y el máximo es 77. Además, todos los valores intermedios se pueden calcular en este problema, como se comprueba en el for que no tiene solución para el valor -70 ni para el valor 77.

## Calcula la complejidad del algoritmo por fuerza bruta

### Respuesta

**Complejidad del algoritmo por fuerza bruta**

El algoritmo `existe_valor_fuerza_bruta()` hace, en esencia, tres bucles anidados:

1. `combinaciones()`: elige qué 5 cifras de las 9 se usan -> $\binom{9}{5}=126$ combinaciones.
2. `permutar(combi)`: para cada combinación, genera **todas** las ordenaciones de esas 5 cifras -> $5!=120$ permutaciones.
3. `permutar(['+', '-', '*', '/'])`: para cada ordenación de cifras, prueba **todas** las ordenaciones de los 4 símbolos -> $4!=24$ permutaciones, evaluando la expresión con `eval`. En este problema concreto, la evaluación de la expresión puede considerarse O(1), ya que siempre tiene longitud fija (5 números y 4 operadores).

Por tanto, el número total de expresiones evaluadas es:

$$\binom{9}{5}\cdot 5!\cdot 4! = 126\cdot120\cdot24 = 362,880$$

que coincide con el recuento combinatorio que hicimos al principio del notebook para "el número de posibilidades teniendo en cuenta las restricciones".

Como cada evaluación es de coste constante (siempre son 5 números y 4 operadores), la complejidad total es:

$$O\Big(\binom{n}{k}\cdot k!\cdot(k-1)!\Big) \quad \text{con } n=9,\ k=5$$

Si generalizamos el problema a "elegir $k$ cifras de un total de $n$", esta expresión crece de forma **factorial**, mucho peor que exponencial: duplicar $k$ multiplica el coste por varios órdenes de magnitud. Es la complejidad típica de un algoritmo de fuerza bruta que no descarta ninguna rama del árbol de posibilidades: genera y evalúa *todo* el espacio de soluciones.

En el caso concreto del problema, el algoritmo evalúa como máximo las 362,880 expresiones posibles. Aunque existe retorno temprano cuando se encuentra el objetivo, en el peor caso debe recorrer todo el espacio de soluciones.

## (*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

### Respuesta

**Técnica elegida: Backtracking con poda (Ramificación y Poda / Branch and Bound)**

*Nota sobre Divide y Vencerás:* en primera instancia se consideró, pero no encaja bien en la parte que realmente cuesta (la búsqueda entre combinaciones). Divide y Vencerás funciona muy bien cuando el problema se puede partir en subproblemas **independientes** cuya solución se combina en menos tiempo del que costaría fuerza bruta (mergesort, el par de puntos más cercano, Strassen...). Aquí no tenemos eso: no podemos elegir "la mitad izquierda" de cifras y operadores por separado y luego combinarla con "la mitad derecha", porque las 5 cifras y los 4 operadores salen del **mismo conjunto sin repetición**, así que las decisiones no son independientes. Donde Divide y Vencerás sí aparece de forma natural es dentro de la propia evaluación de una expresión (un árbol de expresión que se evalúa recursivamente respetando la precedencia de operadores), pero eso no es el cuello de botella del problema: `eval()` ya lo hace en tiempo constante porque la expresión siempre tiene 9 tokens.

El cuello de botella real es la **búsqueda combinatoria**: probar qué cifras, en qué orden, y con qué operadores. Para mejorar eso sin usar Voraz (que no garantizaría encontrar el óptimo, porque decidir "el mejor" operador o cifra en cada paso de forma local no asegura el mejor resultado global), la técnica adecuada es **backtracking con poda** (también llamada ramificación y poda, o Branch and Bound):

- Construímos la expresión token a token (número, operador, número, operador...), respetando la alternancia y sin repetir.
- En cada nodo del árbol de búsqueda en el que ya hay un número recién colocado, evaluamos el prefijo (ej. `"7+8*9"`) y calculamos una **cota**: el máximo aporte, en valor absoluto, que pueden hacer los números que faltan por colocar, sabiendo exactamente cuántos operadores multiplicativos (`*`, `/`) quedan disponibles (si solo queda un `*`/`/`, como mucho puede formarse un término con dos números multiplicados, no un producto gigante de todos los restantes).
- Con esa cota superior e inferior, si **ninguna** rama que sale de este nodo puede mejorar el mejor máximo/mínimo encontrado hasta ahora (o, en la búsqueda de un valor objetivo, si el objetivo ni siquiera cabe en el rango alcanzable), se **poda** la rama: no se sigue explorando.
- A diferencia de un algoritmo voraz, aquí no se descarta ninguna posibilidad que *pudiera* ser la solución: las ramas se descartan cuando las cotas calculadas indican que no pueden conducir al objetivo o mejorar la mejor solución conocida. Por eso el resultado sigue siendo exacto (óptimo garantizado), simplemente evitando explorar partes del árbol que no aportan nada, aunque el peor caso sigue siendo factorial.

In [5]:
import math

NUMEROS = list(range(1, 10))
OPERADORES = ['+', '-', '*', '/']

def cota_extra(numeros_usados, operadores_usados, huecos_numeros_restantes):
    """Cota (segura) de lo máximo que pueden llegar a aportar, en valor absoluto,
    los números que aún faltan por colocar.
    - Solo se cuentan tantos números como huecos quedan realmente (no todas las
      cifras sin usar del 1-9, que sería una cota demasiado floja), utilizando
      los mayores.
    - Los operadores multiplicativos (*, /) que aún queden pueden agrupar varios
      números en un único término grande; el resto de números se suman aparte."""
    disponibles = sorted((n for n in NUMEROS if n not in numeros_usados), reverse=True)[:huecos_numeros_restantes]
    mult_restantes = sum(1 for o in ('*', '/') if o not in operadores_usados)
    tam_termino_grande = min(mult_restantes + 1, len(disponibles))
    termino_grande = 1
    for d in disponibles[:tam_termino_grande]:
        termino_grande *= d
    resto = sum(disponibles[tam_termino_grande:])
    return termino_grande + resto

def se_poda(secuencia, numeros_usados, operadores_usados, estado, objetivo=None):
  """
  Determina si la solución actual se debería podar, es decir, por la cota
  calculada se corta la rama si no es posible mejorar el máximo y el mínimo.
  """
  valor_parcial = eval(''.join(secuencia))
  huecos_restantes = 5 - len(numeros_usados)
  extra = cota_extra(numeros_usados, operadores_usados, huecos_restantes)
  cota_sup, cota_inf = valor_parcial + extra, valor_parcial - extra
  if objetivo is None:
    # PODA: si ni el máximo ni el mínimo pueden mejorar desde aquí, cortamos la rama
    if cota_sup <= estado['max'] and cota_inf >= estado['min']:
      return True
  else:
    # PODA: si el objetivo no cabe en el rango alcanzable, es imposible desde aquí
    if objetivo < cota_inf or objetivo > cota_sup:
      return True
  return False

def backtracking(secuencia, numeros_usados, operadores_usados, estado, objetivo=None):
    """Construye la expresión número-operador-número-... alternando y sin repetir.
    Si objetivo es None: busca el máximo y el mínimo alcanzables.
    Si objetivo tiene un valor: busca una expresión que dé exactamente ese valor."""
    estado['nodos'] += 1
    if objetivo is not None and estado.get('encontrado'):
        return

    if len(secuencia) == 9:
        # La secuencia candidato a solución final o para el cálculo de máximos y mínimos
        valor = eval(''.join(secuencia))
        if objetivo is None:
            if valor > estado['max']:
                estado['max'], estado['max_expr'] = valor, secuencia.copy()
            if valor < estado['min']:
                estado['min'], estado['min_expr'] = valor, secuencia.copy()
        elif valor == objetivo:
            estado['encontrado'] = True
            estado['expr'] = secuencia.copy()
        return

    if len(secuencia) % 2 == 1:
        # el último token colocado es un número: hay un prefijo evaluable
        if se_poda(secuencia, numeros_usados, operadores_usados, estado, objetivo):
          return

        # La solción no se ha podado, continuamos agregando operadores
        for o in OPERADORES:
            if objetivo is not None and estado.get('encontrado'):
                return
            if o not in operadores_usados:
                secuencia.append(o); operadores_usados.add(o)
                backtracking(secuencia, numeros_usados, operadores_usados, estado, objetivo)
                operadores_usados.remove(o); secuencia.pop()

    else:
        # Número par de elementos, agregamos el siguiente número
        for n in NUMEROS:
            if objetivo is not None and estado.get('encontrado'):
                return
            if n not in numeros_usados:
                secuencia.append(str(n)); numeros_usados.add(n)
                backtracking(secuencia, numeros_usados, operadores_usados, estado, objetivo)
                numeros_usados.remove(n); secuencia.pop()

def encontrar_extremos():
    """
    Encuentra el valor mínimo y máximo para todas las expresiones.
    """
    estado = {'max': -float('inf'), 'max_expr': None, 'min': float('inf'), 'min_expr': None, 'nodos': 0}
    backtracking([], set(), set(), estado)
    return estado

def existe_valor(objetivo):
    """
    Dado un valor objetivo que debe estar entre los valores -70 y 78 para que
    tenga solución, dará un diccionario indicando 'encontrado' verdadero o falso
    si se encontró una expresión, la expresión solución y los números de nodos
    visitados.
    """
    estado = {'encontrado': False, 'expr': None, 'nodos': 0}
    backtracking([], set(), set(), estado, objetivo)
    return estado

# --- Máximo y mínimo con poda ---
res = encontrar_extremos()
print(f"Máximo con valor {res['max']} es {''.join(res['max_expr'])}")
print(f"Mínimo con valor {res['min']} es {''.join(res['min_expr'])}")
print(f"Nodos explorados con poda: {res['nodos']}  (fuerza bruta explora 527.374 nodos en el mismo árbol)")

# --- ¿Se pueden alcanzar todos los enteros entre el mínimo y el máximo? ---
inicio, fin = math.ceil(res['min']), math.floor(res['max'])
no_alcanzables = []
nodos_barrido = 0
for objetivo in range(inicio, fin + 1):
    r = existe_valor(objetivo)
    nodos_barrido += r['nodos']
    if not r['encontrado']:
        no_alcanzables.append(objetivo)

print(f"\nEnteros comprobados entre {inicio} y {fin}: {fin - inicio + 1}")
print(f"Enteros NO alcanzables: {no_alcanzables}")
print(f"Nodos explorados en todo el barrido: {nodos_barrido}")

# Ejemplo dado
resultado_ejemplo = existe_valor(4)
sol_ejemplo=''.join(resultado_ejemplo.get('expr'))
print(f"\nEjemplo para valor especifico 4: {sol_ejemplo} revisando {resultado_ejemplo.get('nodos')} nodos")



Máximo con valor 78.83333333333333 es 7+8*9-1/6
Mínimo con valor -70.71428571428571 es 1-8*9+2/7
Nodos explorados con poda: 47492  (fuerza bruta explora 527.374 nodos en el mismo árbol)

Enteros comprobados entre -70 y 78: 149
Enteros NO alcanzables: [-70, 78]
Nodos explorados en todo el barrido: 770308

Ejemplo para valor especifico 4: 1+3*8/2-9 revisando 2201 nodos


Los valores obtenidos coinciden con los de la solución por fuerza bruta, ya que ambos algoritmos resuelven el mismo problema. Sin embargo, el algoritmo de ramificación y poda explora un número menor de nodos del árbol de búsqueda al descartar anticipadamente aquellas ramas que, según las cotas calculadas, no pueden conducir a una solución válida o al valor buscado.

La expresión encontrada puede diferir con el obtenido con fuerza bruta, ya que un mismo valor puede alcanzarse mediante distintas expresiones, es decir, un mismo problema puede tener distintas soluciones. Por ejemplo, 1+3\*8/2-9 = 4 y 4+2-6/3\*1 =4, por lo que cualquiera de ellas constituye una solución correcta..

## (*)Calcula la complejidad del algoritmo

### Respuesta

**Complejidad del algoritmo con poda**

En el **peor caso**, la poda no cambia la cota teórica: si la cota nunca llegara a permitir podar ninguna rama, se exploraría el mismo árbol completo que la fuerza bruta, así que la cota superior asintótica sigue siendo

$$O\Big(\binom{n}{k}\cdot k!\cdot(k-1)!\Big)$$

Esto es así porque la **cota utilizada para la poda no garantiza una reducción del peor caso**. Si la cota apenas permite descartar ramas, el algoritmo puede terminar recorriendo prácticamente el mismo árbol que la fuerza bruta. La eficacia de la poda depende de lo ajustadas que sean las cotas empleadas.

Donde sí se nota la mejora es en el **caso práctico**, y lo hemos medido directamente contando nodos visitados sobre el mismo árbol de búsqueda:

| | Nodos explorados |
|---|---|
| Fuerza bruta (árbol completo) | 527.374 |
| Backtracking con poda (máximo y mínimo) | 47.492 |

Para comparar el algoritmo con poda frente a la fuerza bruta se ha utilizado el mismo algoritmo de backtracking, desactivando temporalmente la llamada a `se_poda()`. De este modo, el algoritmo recorre el árbol completo de búsqueda sin descartar ninguna rama, mientras que el contador de nodos continúa incrementándose en cada llamada recursiva. El resultado obtenido es un total de 527,374 nodos explorados, que representa el número total de nodos del árbol de búsqueda completo.

Es decir, con la poda visitamos aproximadamente el **9%** de los nodos que visitaría la fuerza bruta para encontrar exactamente los mismos valores óptimos. La razón intuitiva es que, en cuanto encontramos una buena solución (máximo o mínimo) cerca de la raíz del árbol, cualquier rama cuyo mejor caso posible no supere ya ese valor se descarta sin necesidad de generar todas sus hojas, y como el árbol crece factorialmente, podar cerca de la raíz ahorra ramas enteras, no solo nodos sueltos.

Para la búsqueda de un valor objetivo concreto la poda es aún más efectiva, porque además de las cotas de rango, la búsqueda **se detiene en cuanto encuentra una solución válida** (no hace falta seguir explorando una vez confirmado que el valor es alcanzable).

## Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

### Respuesta

**Diseño del juego de datos aleatorios**

El "input" propiamente dicho del problema (las cifras 1-9 y los 4 operadores) es siempre el mismo, así que no tiene sentido aleatorizarlo. Lo que sí varía de una ejecución a otra es el **valor objetivo** que le pedimos al algoritmo `existe_valor(objetivo)` que intente alcanzar. Por eso el juego de datos aleatorio consiste en una lista de enteros objetivo, construida para cubrir tres casos distintos y así probar el algoritmo de forma representativa:

- **Valores dentro del rango** [mínimo, máximo] = [-69, 77] representan los casos habituales, en los que el algoritmo debe encontrar una expresión válida.
- **Valores fuera de rango** (menores que -69 o mayores que 77): el algoritmo debería descartarlos rápidamente gracias a la poda por cota.
- **Casos de control**: los dos valores límite -70 y 78, que se sabe que no son alcanzables exactamente, para comprobar que el algoritmo los sigue clasificando correctamente como "no encontrado".

In [6]:
#Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios
import random
random.seed(42)  # semilla fija para que el experimento sea reproducible

minimo_int, maximo_int = -69, 77  # extremos enteros ya calculados en el apartado de fuerza bruta

dentro_de_rango = random.sample(range(minimo_int, maximo_int + 1), 10)
fuera_de_rango = random.sample(list(range(-120, -80)) + list(range(90, 130)), 5)
casos_control = [-70, 78]

juego_de_datos = sorted(set(dentro_de_rango + fuera_de_rango + casos_control))
print("Juego de datos generado:", juego_de_datos)

Juego de datos generado: [-117, -116, -109, -93, -91, -70, -63, -47, -43, -41, -34, -12, -7, 1, 39, 70, 78]


##Aplica el algoritmo al juego de datos generado

###Respuesta

**Resultado de aplicar el algoritmo al juego de datos**

Para cada valor objetivo del dataset llamamos a `existe_valor(objetivo)` (la versión con poda) y anotamos si se ha encontrado una expresión válida, cuál es, y cuántos nodos ha tenido que explorar el backtracking para decidirlo.

Se observa justo el comportamiento esperado:
- Los valores fuera de [-69, 77] (por ejemplo -117 o 130) se descartan explorando relativamente pocos nodos, porque la cota detecta pronto que son inalcanzables.
- Los dos casos de control (-70 y 78) se clasifican correctamente como **no alcanzables**, aunque exploran más nodos que el resto de "NO" porque están justo en el borde del rango (la cota no puede descartarlos tan rápido como a un valor claramente fuera de rango, al encontrar se próximos al intervalo de resultados alcanzables).
- El resto de valores dentro de rango sí encuentran una expresión válida, y en general con muy pocos nodos explorados en comparación con los 527.374 nodos de la fuerza bruta.

Estos resultados confirman que la poda resulta especialmente efectiva cuando el objetivo es claramente inalcanzable o cuando encuentra una solución en las primeras ramas exploradas, reduciendo significativamente el número de nodos visitados respecto a la exploración exhaustiva.

In [7]:
#Aplica el algoritmo al juego de datos generado
for objetivo in juego_de_datos:
    r = existe_valor(objetivo)
    estado_txt = f"SI -> {''.join(r['expr'])}" if r['encontrado'] else "NO"
    print(f"objetivo={objetivo:>5} | alcanzable={estado_txt:28} | nodos={r['nodos']}")

objetivo= -117 | alcanzable=NO                           | nodos=3790
objetivo= -116 | alcanzable=NO                           | nodos=3790
objetivo= -109 | alcanzable=NO                           | nodos=4238
objetivo=  -93 | alcanzable=NO                           | nodos=5582
objetivo=  -91 | alcanzable=NO                           | nodos=5750
objetivo=  -70 | alcanzable=NO                           | nodos=18874
objetivo=  -63 | alcanzable=SI -> 2-8*9+7/1              | nodos=8743
objetivo=  -47 | alcanzable=SI -> 2+5-6*9/1              | nodos=9580
objetivo=  -43 | alcanzable=SI -> 2+3-6*8/1              | nodos=9463
objetivo=  -41 | alcanzable=SI -> 1-5*9+6/2              | nodos=3705
objetivo=  -34 | alcanzable=SI -> 2+4-5*8/1              | nodos=10763
objetivo=  -12 | alcanzable=SI -> 1-2*8+9/3              | nodos=4613
objetivo=   -7 | alcanzable=SI -> 1+2-5/3*6              | nodos=84
objetivo=    1 | alcanzable=SI -> 1+2-3/6*4              | nodos=31
objetivo=   39 | alcan

##Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

- Documentación oficial de Python: funciones `eval()`, `itertools` y `random`. https://docs.python.org/3/
- Cormen, T. H., Leiserson, C. E., Rivest, R. L., & Stein, C. — *Introduction to Algorithms* (capítulos sobre backtracking y ramificación y poda / branch and bound), como referencia general de la técnica.
- Apuntes y material docente del seminario de la asignatura "Algoritmos de Optimización".

##Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

###Respuesta

**Posibles líneas de continuación**

- **Generalizar el problema**: permitir $n$ cifras disponibles y $k$ cifras a usar (en vez de fijar 9 y 5), y comprobar cómo escala en la práctica el número de nodos podados frente al total del árbol a medida que crece $k$.
- **Cotas más ajustadas**: nuestra cota de poda es deliberadamente conservadora (asume que cualquier término nuevo se puede sumar en positivo, aunque en ese punto de la búsqueda solo quede el operador "-" disponible). Una cota que tenga en cuenta exactamente qué operador aditivo concreto queda libre podaría bastantes más ramas.
- **Investigar por qué -70 y 78 no son alcanzables**: es un resultado curioso que ha salido de la propia experimentación; estudiar analíticamente la estructura de las expresiones cercanas al máximo y al mínimo podría explicar por qué esos dos enteros concretos quedan fuera y los demás no.
- **Ampliar el conjunto de operadores** (potencia, módulo, raíz) o **relajar la restricción de alternancia estricta** cifra-operador-cifra, y estudiar cómo cambia el espacio de soluciones.
- **Paralelizar la búsqueda**: a partir de cierta profundidad, las distintas ramas del árbol de backtracking son independientes entre sí, por lo que se podrían repartir entre varios procesos o hilos.
- **Metaheurísticas para tamaños grandes**: si el problema se generaliza a valores de $n$ y $k$ mucho mayores, la búsqueda exacta (aunque tenga poda) puede dejar de ser viable en tiempo razonable; en ese escenario tendría sentido explorar algoritmos genéticos, recocido simulado o búsqueda tabú para obtener buenas soluciones aproximadas, asumiendo que ya no se garantizaría el óptimo exacto.